# Module 5 — Inverse Kinematics
## Unit 6 — Singularities and Solution Selection
### Lesson 6.4 — Singularities and Solution Selection (Unit 6 Recap)

*Physical AI Curriculum · hands-on notebook. Run **Kernel → Restart & Run All**.*


## Unit 6 synthesis
solve → recognize singularity → filter limits → choose nearest.


In [ ]:
import numpy as np
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt

def fk_two_link(t1, t2, L1, L2):
    return np.array([L1*np.cos(t1)+L2*np.cos(t1+t2), L1*np.sin(t1)+L2*np.sin(t1+t2)])

def jacobian_2link(t1, t2, L1, L2):
    s1,s12=np.sin(t1),np.sin(t1+t2); c1,c12=np.cos(t1),np.cos(t1+t2)
    return np.array([[-L1*s1-L2*s12, -L2*s12],[L1*c1+L2*c12, L2*c12]])

def ik_2link_closed(x, y, L1, L2):
    c2=(x*x+y*y-L1*L1-L2*L2)/(2*L1*L2)
    if c2<-1-1e-9 or c2>1+1e-9: return []
    c2=max(-1.0,min(1.0,c2)); sols=[]
    for sign in (+1.0,-1.0):
        s2=sign*np.sqrt(max(0.0,1-c2*c2)); t2=np.arctan2(s2,c2)
        t1=np.arctan2(y,x)-np.arctan2(L2*np.sin(t2),L1+L2*np.cos(t2))
        sols.append((t1,t2))
        if abs(s2)<1e-6: break
    return sols

L1,L2=0.4,0.3
def norm180(a): return (a+180)%360-180
def pipeline(target, theta_cur, limits):
    sols=ik_2link_closed(*target,L1,L2)
    if not sols: return None,"unreachable"
    feas=[(t1,t2) for (t1,t2) in sols
          if limits[0][0]<=norm180(np.degrees(t1))<=limits[0][1]
          and limits[1][0]<=norm180(np.degrees(t2))<=limits[1][1]]
    if not feas: return None,"no feasible solution"
    best=min(feas,key=lambda s:np.hypot(norm180(np.degrees(s[0])-np.degrees(theta_cur[0])),
                                        norm180(np.degrees(s[1])-np.degrees(theta_cur[1]))))
    detJ=L1*L2*np.sin(best[1])
    return best,("near-singular" if abs(detJ)<0.02 else "ok")
best,status=pipeline([0.5,0.0],np.radians([-30,80]),[(-45,45),(0,150)])
print("chosen",tuple(np.round(np.degrees(best),2)),"status:",status)


## Check your work


In [ ]:
best,status=pipeline([0.5,0.0],np.radians([-30,80]),[(-45,45),(0,150)])
assert best is not None and abs(np.degrees(best[1])-90)<1e-6 and status=="ok"
none,s2=pipeline([0.9,0.0],np.radians([0,0]),[(-180,180),(-180,180)])
assert none is None and s2=="unreachable"
print("All checks passed.")
